# Step 2 - Translate MedEV (Vietnamese -> English)

Translate MedEV source Vietnamese sentences to English using two models via OpenRouter:
- GPT: `openai/gpt-5.2`
- Gemini: `google/gemini-2.5-flash`

Outputs:
- `outputs_medev/translations_gpt.csv`
- `outputs_medev/translations_gemini.csv`

In [87]:
import os
import re
import time
import pandas as pd
from openai import OpenAI

DATA_PATH = os.path.join("data", "medev_test.csv")
OUT_DIR = "outputs_medev"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_GPT = os.path.join(OUT_DIR, "translations_gpt.csv")
OUT_GEMINI = os.path.join(OUT_DIR, "translations_gemini.csv")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in environment.")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

MODELS = {
    "gpt": "openai/gpt-5.2",
    "gemini": "google/gemini-2.5-flash",
}

MAX_TOKENS = 192      # starting max_tokens
MIN_TOKENS = 48       # fallback floor when credits are low
REQUEST_DELAY = 0.2
RETRY_ERROR_ROWS = True
MAX_CONSECUTIVE_402 = 10

medev = pd.read_csv(DATA_PATH)
print(f"Loaded rows: {len(medev):,}")
medev.head(3)

Loaded rows: 8,959


,id,source_vi,reference_en
0,1,Thực trạng kiến thức và thực hành của người có...,"Knowledge, practices in public health service ..."
1,2,"Mô tả thực trạng kiến thức, thực hành của ngườ...","Describe knowledge, practices in public health..."
2,3,Phương pháp: Thiết kế nghiên mô tả cắt ngang đ...,Methodology: A cross sectional study was used ...


In [88]:
SYSTEM_PROMPT = (
    "You are a professional medical translator. "
    "Translate the Vietnamese medical sentence into natural and accurate English. "
    "Return only the translated sentence with no explanation."
)


def is_402_error_text(s: str) -> bool:
    t = str(s).lower()
    return ("error code: 402" in t) or ("'code': 402" in t) or (" code 402" in t)


def _extract_affordable_tokens(msg: str):
    flat = " ".join(str(msg).split())
    m = re.search(r"can only afford\s+(\d+)", flat, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def translate_sentence(text: str, model_id: str, retries: int = 4) -> str:
    adaptive_max_tokens = MAX_TOKENS

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": str(text)},
                ],
                temperature=0,
                max_tokens=adaptive_max_tokens,
            )

            content = response.choices[0].message.content
            if content is None or str(content).strip() == "":
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return "ERROR: Empty response content"

            return str(content).strip()

        except Exception as e:
            msg = str(e)

            if is_402_error_text(msg):
                affordable = _extract_affordable_tokens(msg)
                if affordable is not None and attempt < retries - 1:
                    # Reduce tokens and retry once more before giving up
                    new_max = max(MIN_TOKENS, min(adaptive_max_tokens - 16, affordable - 8))
                    if new_max < adaptive_max_tokens:
                        adaptive_max_tokens = new_max
                        time.sleep(1)
                        continue
                return f"ERROR_402: {msg}"

            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"ERROR: {msg}"


print("translate_sentence() ready")

translate_sentence() ready


In [89]:
def run_translation(model_key: str, output_path: str):
    model_id = MODELS[model_key]

    if os.path.exists(output_path):
        done = pd.read_csv(output_path)

        # Normalize legacy 402 errors from old runs so reporting is consistent
        mask_legacy_402 = done["translated_en"].astype(str).apply(is_402_error_text)
        done.loc[mask_legacy_402, "translated_en"] = done.loc[mask_legacy_402, "translated_en"].astype(str).apply(
            lambda x: x if str(x).startswith("ERROR_402") else f"ERROR_402: {x}"
        )

        if RETRY_ERROR_ROWS:
            ok_mask = ~done["translated_en"].astype(str).str.startswith("ERROR", na=False)
            done_ids = set(done.loc[ok_mask, "id"].tolist())
            print(f"[{model_key}] Resuming: {len(done):,} rows found, {len(done_ids):,} successful rows kept")
        else:
            done_ids = set(done["id"].tolist())
            print(f"[{model_key}] Resuming: {len(done):,} done rows")
    else:
        done = pd.DataFrame(columns=["id", "source_vi", "reference_en", "translated_en", "model"])
        done_ids = set()

    rows = []
    total = len(medev)
    consecutive_402 = 0

    for i, r in medev.iterrows():
        sid = int(r["id"])
        if sid in done_ids:
            continue

        pred = translate_sentence(r["source_vi"], model_id)
        if str(pred).startswith("ERROR_402"):
            consecutive_402 += 1
        else:
            consecutive_402 = 0

        rows.append(
            {
                "id": sid,
                "source_vi": r["source_vi"],
                "reference_en": r["reference_en"],
                "translated_en": pred,
                "model": model_key,
            }
        )

        if (i + 1) % 25 == 0:
            print(f"  [{model_key}] {i + 1}/{total}")

        if consecutive_402 >= MAX_CONSECUTIVE_402:
            print(f"[{model_key}] Stopping early due to consecutive 402 errors (likely insufficient credits).")
            break

        time.sleep(REQUEST_DELAY)

    new_df = pd.DataFrame(rows)
    final_df = pd.concat([done, new_df], ignore_index=True)
    final_df = final_df.sort_values("id").drop_duplicates(subset=["id"], keep="last").reset_index(drop=True)
    final_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"[{model_key}] Saved: {len(final_df):,} -> {output_path}")
    return final_df


print("run_translation() ready")

run_translation() ready


In [90]:
print("=" * 60)
print("Translating with GPT...")
print("=" * 60)
gpt_df = run_translation("gpt", OUT_GPT)
gpt_df.head(5)

Translating with GPT...
[gpt] Resuming: 8,959 rows found, 8,958 successful rows kept
[gpt] Saved: 8,959 -> outputs_medev\translations_gpt.csv


,id,source_vi,reference_en,translated_en,model
0,1,Thực trạng kiến thức và thực hành của người có...,"Knowledge, practices in public health service ...",The current status of knowledge and practices ...,gpt
1,2,"Mô tả thực trạng kiến thức, thực hành của ngườ...","Describe knowledge, practices in public health...",Describe the current status of knowledge and p...,gpt
2,3,Phương pháp: Thiết kế nghiên mô tả cắt ngang đ...,Methodology: A cross sectional study was used ...,Methods: A cross-sectional descriptive study w...,gpt
3,4,Kết quả: Tỷ lệ người biết được khám chữa bệnh ...,Results: Percentage of card's holders who knew...,Results: The proportion of people who were awa...,gpt
4,5,Tỷ lệ người có thẻ BHYT thực hành khám chữa bệ...,Percentage of card's holders who went to the f...,The proportion of health insurance cardholders...,gpt


In [91]:
print("=" * 60)
print("Translating with Gemini...")
print("=" * 60)
gemini_df = run_translation("gemini", OUT_GEMINI)
gemini_df.head(5)

Translating with Gemini...
[gemini] Resuming: 8,959 rows found, 8,956 successful rows kept
[gemini] Saved: 8,959 -> outputs_medev\translations_gemini.csv


,id,source_vi,reference_en,translated_en,model
0,1,Thực trạng kiến thức và thực hành của người có...,"Knowledge, practices in public health service ...",Current status of knowledge and practices of h...,gemini
1,2,"Mô tả thực trạng kiến thức, thực hành của ngườ...","Describe knowledge, practices in public health...",Describe the current status of knowledge and p...,gemini
2,3,Phương pháp: Thiết kế nghiên mô tả cắt ngang đ...,Methodology: A cross sectional study was used ...,Methods: A cross-sectional descriptive study w...,gemini
3,4,Kết quả: Tỷ lệ người biết được khám chữa bệnh ...,Results: Percentage of card's holders who knew...,Results: The rate of people who knew about fre...,gemini
4,5,Tỷ lệ người có thẻ BHYT thực hành khám chữa bệ...,Percentage of card's holders who went to the f...,The rate of people with health insurance cards...,gemini


In [92]:
for label, df in [("GPT", gpt_df), ("Gemini", gemini_df)]:
    print(f"\n--- {label} ---")
    print(f"Rows: {len(df):,}")

    err_mask = df["translated_en"].astype(str).str.startswith("ERROR", na=False)
    err_402_mask = df["translated_en"].astype(str).apply(is_402_error_text)

    errors_all = df[err_mask]
    errors_402 = df[err_402_mask]

    print(f"Errors total: {len(errors_all):,} | 402 errors: {len(errors_402):,}")

    if len(errors_402):
        print("Sample 402 messages:")
        display(errors_402[["id", "translated_en"]].head(3))

    display(df.sample(min(3, len(df)), random_state=42))


--- GPT ---
Rows: 8,959
Errors total: 1 | 402 errors: 1
Sample 402 messages:


,id,translated_en
6850,6851,ERROR_402: ERROR: Error code: 402 - {'error': ...


,id,source_vi,reference_en,translated_en,model
5712,5713,Kháng sinh dự phòng không cần thiết cho DPL.,Prophylactic antibiotics are not needed for DPL.,Prophylactic antibiotics are not necessary for...,gpt
7642,7643,Khi phát hiện có thương tổn hạch bạch huyết tr...,When mediastinal lymph node involvement is fou...,When mediastinal lymph node involvement is ide...,gpt
8562,8563,- PCEA có liều nền 2 - 4 ml / giờ mang lại sự ...,- Satisfaction for women of PCEA groups with b...,PCEA with a basal infusion of 2–4 mL/hour prov...,gpt



--- Gemini ---
Rows: 8,959
Errors total: 3 | 402 errors: 3
Sample 402 messages:


,id,translated_en
709,710,ERROR_402: ERROR: Error code: 402 - {'error': ...
801,802,ERROR_402: ERROR: Error code: 402 - {'error': ...
880,881,ERROR_402: ERROR: Error code: 402 - {'error': ...


,id,source_vi,reference_en,translated_en,model
5712,5713,Kháng sinh dự phòng không cần thiết cho DPL.,Prophylactic antibiotics are not needed for DPL.,Prophylactic antibiotics are not necessary for...,gemini
7642,7643,Khi phát hiện có thương tổn hạch bạch huyết tr...,When mediastinal lymph node involvement is fou...,When mediastinal lymph node involvement is det...,gemini
8562,8563,- PCEA có liều nền 2 - 4 ml / giờ mang lại sự ...,- Satisfaction for women of PCEA groups with b...,- PCEA with a background infusion of 2 - 4 ml/...,gemini


In [93]:
# Retry only a few remaining 402 rows (targeted fix)

RETRY_IDS = {
    "gpt": [6851],
    "gemini": [710, 802, 881],
}


def translate_sentence_rescue(text: str, model_id: str, retries: int = 6) -> str:
    # Start very low to fit remaining credit budget
    rescue_tokens = [96, 80, 64, 56, 48, 40]

    for i in range(min(retries, len(rescue_tokens))):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": str(text)},
                ],
                temperature=0,
                max_tokens=rescue_tokens[i],
            )
            content = response.choices[0].message.content
            if content is None or str(content).strip() == "":
                time.sleep(1)
                continue
            return str(content).strip()
        except Exception as e:
            msg = str(e)
            if is_402_error_text(msg):
                time.sleep(1)
                continue
            return f"ERROR: {msg}"

    return "ERROR_402: rescue failed after low-token retries"


def retry_specific_ids(model_key: str, output_path: str, ids: list[int]):
    if not ids:
        return pd.DataFrame()

    out_df = pd.read_csv(output_path)
    model_id = MODELS[model_key]

    # Source rows from medev table
    src = medev[medev["id"].isin(ids)].copy()
    if len(src) == 0:
        print(f"[{model_key}] No matching source rows for IDs: {ids}")
        return out_df

    updates = 0
    for _, r in src.iterrows():
        sid = int(r["id"])
        pred = translate_sentence_rescue(r["source_vi"], model_id)

        out_df.loc[out_df["id"] == sid, "translated_en"] = pred
        out_df.loc[out_df["id"] == sid, "model"] = model_key
        updates += 1

    out_df = out_df.sort_values("id").drop_duplicates(subset=["id"], keep="last").reset_index(drop=True)
    out_df.to_csv(output_path, index=False, encoding="utf-8")

    err_mask = out_df["translated_en"].astype(str).str.startswith("ERROR", na=False)
    err_402_mask = out_df["translated_en"].astype(str).apply(is_402_error_text)
    print(f"[{model_key}] Updated {updates} row(s). Remaining errors: {err_mask.sum()} | 402: {err_402_mask.sum()}")

    return out_df


# Run targeted retries
gpt_df = retry_specific_ids("gpt", OUT_GPT, RETRY_IDS["gpt"])
gemini_df = retry_specific_ids("gemini", OUT_GEMINI, RETRY_IDS["gemini"])

# Show latest status for those IDs
print("\nCheck targeted IDs after retry:")
for mk, ids in RETRY_IDS.items():
    path = OUT_GPT if mk == "gpt" else OUT_GEMINI
    df_check = pd.read_csv(path)
    display(df_check[df_check["id"].isin(ids)][["id", "translated_en"]])

[gpt] Updated 1 row(s). Remaining errors: 0 | 402: 0
[gemini] Updated 3 row(s). Remaining errors: 0 | 402: 0

Check targeted IDs after retry:


,id,translated_en
6850,6851,"In systemic JIA, the rash may be diffuse and m..."


,id,translated_en
709,710,"Conclusion: Intraventricular conduction delay,..."
801,802,The mean follow-up period was 11.6 ± 6.4 months.
880,881,Conclusion: The level of patient satisfaction ...
